In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys & Initialize the LLM
# ============================================================================
# We use python-dotenv to securely load API keys from a .env file
# This is a best practice - never hardcode API keys in your notebooks!
# ============================================================================

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print("✅ Environment variables loaded successfully!")
print(f"🤖 LLM initialized: {llm.model_name}")

# ----------------------------------------------------------------------------
# PREVIOUS SETUP (kept for reference): platform-aware `helpers` factory
# ----------------------------------------------------------------------------
# # ============================================================================
# # ENVIRONMENT SETUP: Load API Keys & Import Dependencies
# # ============================================================================
# # We use python-dotenv to securely load API keys from a .env file
# # This is a best practice - never hardcode API keys in your notebooks!
# # ============================================================================
#
# from dotenv import load_dotenv
# import os
# import sys
# import platform
#
# # Load environment variables from .env file
# load_dotenv()
#
# # Add parent directory to path for importing helpers
# sys.path.append(os.path.abspath("../.."))
#
# # Import our LLM factory functions
# # - get_groq_llm(): Creates a Groq-hosted LLM (fast inference with open-source models)
# # - get_openai_llm(): Creates an OpenAI GPT model
# # - get_databricks_llm(): Creates a Databricks-hosted LLM
# from helpers.utils import get_groq_llm, get_openai_llm, get_databricks_llm
#
# print("✅ Environment variables loaded successfully!")
# print(f"📍 Running on: {platform.system()}")
#
# # -----------------------------------------------------------------------------
# # Initialize the LLM based on platform or preference
# # The choice of LLM affects tool calling capabilities and speed
# # -----------------------------------------------------------------------------
# if sys.platform == "win32":
#     # Windows: Use Groq for fast inference
#     llm = get_groq_llm()
# elif sys.platform == "darwin":
#     # macOS: Use Databricks-hosted Gemini
#     llm = get_databricks_llm("databricks-gemini-2-5-pro")  
# else:
#     # Linux: Default to Groq
#     llm = get_groq_llm()
#
# # Print which LLM we're using
# if hasattr(llm, 'model_name'):
#     print(f"🤖 LLM initialized: {llm.model_name}")
# elif hasattr(llm, 'model'):
#     print(f"🤖 LLM initialized: {llm.model}")
# else:
#     print("🤖 LLM initialized successfully")

### Introduction of LangChain

In [ ]:
try:
    # Note: llm is already initialized in the setup cell using helper functions
    # The llm object is created based on your platform (Windows: Groq, macOS: Databricks, Linux: Groq)

    # Get the response
    response = llm.invoke('What is the capital of France?')
    print(response.content)
    
    print("-------------------------------------------------")
    
    # Use llm.invoke() with another question
    alt_response = llm.invoke("What is LangChain? Explain in 1 line")
    print(alt_response.content)
except Exception as e:
    print(e)

### First LangChain Project 

A LangChain pipeline is a sequence of steps where inputs are processed to produce outputs. For your first project, you’ll create a basic pipeline that uses an LLM to generate text responses based on user input.



Build a pipeline that: 
* Accepts a user’s question as input.  
* Uses a pre-trained LLM to generate a response.   
* Outputs the generated text.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
try:
    
    # Step 1: Create a Prompt Template
    prompt = PromptTemplate(
        input_variables=['topic'],
        template="Provide me details about {topic}?"
    )

    # Step 2: Create a Chain using LCEL (LangChain Expression Language)
    # The | operator chains: prompt -> llm -> output parser
    chain = prompt | llm | StrOutputParser()

    # Step 3: Generate the response
    response = chain.invoke({'topic': 'Generative AI'})
    resp = chain.invoke({'topic': 'Deep Learning'})
    
    print(response)
    print("-------------------------------------------------")
    print(resp)
    
except Exception as e:
    print(e)

### Modify the above project
* After generating the response, summarize it into a shorter version.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

try:
    
    # Step 1: Create Prompt Templates
    detail_prompt = PromptTemplate(
        input_variables=['topic'],
        template="Provide me details about {topic}?"
    )

    summary_prompt = PromptTemplate(
        input_variables=['text'],
        template="Summarize the following text into a single concise sentence: {text}"
    )

    # Step 2: Create individual chains using LCEL
    answer_chain = detail_prompt | llm | StrOutputParser()
    summary_chain = summary_prompt | llm | StrOutputParser()

    # Step 3: Combine the Chains using LCEL
    # First chain generates details, then passes to summary chain
    pipeline = (
        {"topic": RunnablePassthrough()}
        | answer_chain
        | (lambda text: {"text": text})
        | summary_chain
    )

    # Step 4: Generate the Response
    response = pipeline.invoke('Deep Learning')

    print(response)
        
    
except Exception as e:
    print(e)

### Add a Sentiment Analysis Step at the end

In [ ]:
# Create a sentiment analysis prompt
sentiment_prompt = PromptTemplate(
    input_variables=['text'],
    template="Analyze the sentiment of the following text: {text}"
)

# Create the sentiment chain
sentiment_chain = sentiment_prompt | llm | StrOutputParser()

# Combine all three chains: details -> summary -> sentiment
full_pipeline = (
    {"topic": RunnablePassthrough()}
    | answer_chain
    | (lambda text: {"text": text})
    | summary_chain
    | (lambda text: {"text": text})
    | sentiment_chain
)

response = full_pipeline.invoke('Deep Learning')

print(response)

### Add a Dynamic User Input Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Create Prompt Template
detail_prompt = PromptTemplate(
    input_variables=['topic'],
    template="Provide me details about {topic}?"
)

# Step 2: Create the chain using LCEL
answer_chain = detail_prompt | llm | StrOutputParser()

while True:
    user_input = input("Enter the topic you want to know more about (or type 'exit' to quit): ")
    
    # Exit the loop if the user types 'exit'
    if user_input.lower() == 'exit':
        print("Exiting the assistant. GoodBye!!")
        break
    
    response = answer_chain.invoke({'topic': user_input})
    print(response)